# FIX NOTEBOOK: Sanskrit Classification — Post-Training Corrections

**Purpose:** Fix bugs in analysis cells WITHOUT retraining any models.  
**Input Required:** `experiment_results_complete.json` from original notebook  
**Time to Run:** < 1 minute (no GPU needed)  

### What This Fixes:
1. **Per-class P=R=F1 bug** — Original used `average='micro'` on single-class filtered data → P=R=F1=accuracy. Fixed by recomputing from saved confusion matrices.
2. **Bootstrap CI bug** — Cross-tokenizer prediction misalignment produced garbage CIs. Fixed by restricting to same-tokenizer models only.
3. **McNemar cross-tokenizer** — Now restricted to valid same-tokenizer pairs only (paper already handled this correctly).
4. **Saves corrected JSON** — `experiment_results_FIXED.json`

In [ ]:
# =============================================================================
# CELL 1: LOAD SAVED RESULTS (NO TRAINING NEEDED)
# =============================================================================

import json
import numpy as np
import os
from scipy.stats import chi2
from sklearn.metrics import precision_score, recall_score, f1_score

# ---------- CONFIGURE THIS PATH ----------
RESULTS_FILE = "experiment_results_complete.json"
# ------------------------------------------

assert os.path.exists(RESULTS_FILE), f"ERROR: '{RESULTS_FILE}' not found! Place it in the same directory as this notebook."

with open(RESULTS_FILE, 'r') as f:
    complete_results = json.load(f)

ALL_RESULTS = complete_results['4_results']
PREDICTIONS = complete_results['5_predictions']

# Label mappings
label2id = {"SUB": 0, "TIN": 1, "OTH": 2}
id2label = {0: "SUB", 1: "TIN", 2: "OTH"}

# Which models share MuRIL tokenizer (valid for cross-model comparison)
MURIL_TOKENIZER_MODELS = {'BiLSTM', 'SuffixCNN', 'MuRIL'}
MURIL_TOKENIZER_MODELS.update(n for n in ALL_RESULTS if n.startswith('CNN-SL'))

# Helper functions
def get_mean(model_name, metric):
    return ALL_RESULTS[model_name]['mean_std'][f'{metric}_mean']

def get_ms(model_name, metric):
    return ALL_RESULTS[model_name]['mean_std'][metric]

# Find best hybrid
best_hybrid_name = max(
    [n for n in ALL_RESULTS if n.startswith('CNN-SL')],
    key=lambda n: get_mean(n, 'f1')
)

key_models = ['BiLSTM', 'SuffixCNN', 'MuRIL', 'XLM-R', 'mBERT', 'IndicBERT', 'ByT5', best_hybrid_name]

print(f"\u2713 Loaded {len(ALL_RESULTS)} models from {RESULTS_FILE}")
print(f"\u2713 Best hybrid: {best_hybrid_name}")
print(f"\u2713 MuRIL-tokenizer models: {len(MURIL_TOKENIZER_MODELS)}")
print(f"\u2713 Cross-tokenizer models: XLM-R, mBERT, IndicBERT, ByT5")
print(f"\nReady to fix!")

AssertionError: ERROR: 'experiment_results_complete.json' not found! Place it in the same directory as this notebook.

In [ ]:
# =============================================================================
# CELL 2: FIX #1 — PER-CLASS METRICS (from confusion matrices)
# =============================================================================
#
# ORIGINAL BUG: Used precision_score(cls_true, cls_preds, average='micro')
# on data filtered to a single class → micro P = micro R = micro F1 = accuracy.
#
# FIX: Recompute P, R, F1 per class from the saved confusion matrix.
# Confusion matrix is [3x3]: rows=true, cols=predicted, labels=[SUB,TIN,OTH]
# =============================================================================

def compute_per_class_from_cm(cm):
    """Compute per-class P, R, F1 from a 3x3 confusion matrix.

    cm[i][j] = count of samples with true label i predicted as j
    Labels: 0=SUB, 1=TIN, 2=OTH
    """
    cm = np.array(cm)
    per_class = {}

    for cls_id in range(3):
        cls_name = id2label[cls_id]
        tp = cm[cls_id, cls_id]
        fp = cm[:, cls_id].sum() - tp      # other classes predicted as this class
        fn = cm[cls_id, :].sum() - tp      # this class predicted as other classes
        support = cm[cls_id, :].sum()       # total true samples of this class

        precision = (tp / (tp + fp) * 100) if (tp + fp) > 0 else 0.0
        recall    = (tp / (tp + fn) * 100) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

        per_class[cls_name] = {
            'precision': round(precision, 2),
            'recall': round(recall, 2),
            'f1': round(f1, 2),
            'support': int(support)
        }

    return per_class

# Apply fix to all models, all seeds
fix_count = 0
for model_name in ALL_RESULTS:
    for key in ALL_RESULTS[model_name]:
        if key.startswith('seed_'):
            seed_data = ALL_RESULTS[model_name][key]
            if 'confusion_matrix' in seed_data:
                old_pc = seed_data.get('per_class', {})
                new_pc = compute_per_class_from_cm(seed_data['confusion_matrix'])
                seed_data['per_class'] = new_pc
                fix_count += 1

print(f"\u2713 Fixed per-class metrics for {fix_count} model-seed combinations")

# Show before vs after for one example
print(f"\n--- Verification: BiLSTM best seed ---")
bs = ALL_RESULTS['BiLSTM']['best_seed']
pc = ALL_RESULTS['BiLSTM'][bs]['per_class']
cm = np.array(ALL_RESULTS['BiLSTM'][bs]['confusion_matrix'])
print(f"Confusion Matrix:\n{cm}\n")
print(f"{'Class':<8} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print("-" * 52)
for cls in ['SUB', 'TIN', 'OTH']:
    c = pc[cls]
    print(f"{cls:<8} {c['precision']:>9.2f}% {c['recall']:>9.2f}% {c['f1']:>9.2f}% {c['support']:>10,}")

print(f"\n(If P \u2260 R for any class, the fix is working correctly!)")

In [ ]:
# =============================================================================
# CELL 3: PRINT CORRECTED TABLE 9 — PER-CLASS PERFORMANCE
# =============================================================================
# Paper Table 9: Per-class F1 for key models (best seed)
# Now with CORRECT separate P, R, F1 values
# =============================================================================

print("=" * 85)
print("CORRECTED TABLE 9: PER-CLASS PERFORMANCE (best seed)")
print("=" * 85)

# Header for compact table (paper format - F1 only)
print(f"\n{'Model':<22} {'SUB F1':>10} {'TIN F1':>10} {'OTH F1':>10}")
print("-" * 55)

for name in key_models:
    if name in ALL_RESULTS:
        bs = ALL_RESULTS[name]['best_seed']
        pc = ALL_RESULTS[name][bs]['per_class']
        print(f"{name:<22} {pc['SUB']['f1']:>9.2f}% {pc['TIN']['f1']:>9.2f}% {pc['OTH']['f1']:>9.2f}%")

# Full detail table
print(f"\n\n{'=' * 85}")
print("DETAILED PER-CLASS METRICS (P, R, F1 — all now distinct)")
print("=" * 85)

for name in key_models:
    if name in ALL_RESULTS:
        bs = ALL_RESULTS[name]['best_seed']
        pc = ALL_RESULTS[name][bs]['per_class']
        print(f"\n{name} (seed: {bs}):")
        print(f"  {'Class':<8} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
        print(f"  {'-' * 50}")
        for cls in ['SUB', 'TIN', 'OTH']:
            c = pc[cls]
            print(f"  {cls:<8} {c['precision']:>9.2f}% {c['recall']:>9.2f}% {c['f1']:>9.2f}% {c['support']:>10,}")

In [ ]:
# =============================================================================
# CELL 4: FIX #2 — McNEMAR TEST (same-tokenizer only)
# =============================================================================
#
# ORIGINAL BUG: Ran McNemar across ALL model pairs including cross-tokenizer
# (XLM-R, mBERT, IndicBERT, ByT5 use different tokenizers → different token
# counts → misaligned predictions → meaningless chi-squared values).
#
# FIX: Only compare models sharing MuRIL tokenizer.
# Paper already did this correctly — this just fixes the notebook.
# =============================================================================

true_labels = np.array(PREDICTIONS['true_labels'])

# Only valid comparisons: models using MuRIL tokenizer
valid_models = [m for m in key_models if m in MURIL_TOKENIZER_MODELS]

print("=" * 80)
print("CORRECTED TABLE 10: McNEMAR TEST (same-tokenizer pairs only)")
print("=" * 80)
print(f"\nValid models (MuRIL tokenizer): {valid_models}")
print(f"Excluded (different tokenizer):  {[m for m in key_models if m not in MURIL_TOKENIZER_MODELS]}\n")

valid_pairs = []
for i, m1 in enumerate(valid_models):
    for m2 in valid_models[i+1:]:
        valid_pairs.append((m1, m2))

n_tests = len(valid_pairs)
bonferroni_alpha = 0.05 / max(n_tests, 1)

print(f"{'Comparison':<40} {'\u03c7\u00b2':>10} {'p-value':>12} {'Sig?':>8}")
print("-" * 75)

mcnemar_results = {}
for m1, m2 in valid_pairs:
    if m1 in PREDICTIONS and m2 in PREDICTIONS:
        pred1 = np.array(PREDICTIONS[m1])
        pred2 = np.array(PREDICTIONS[m2])

        # Same tokenizer → same length, but verify
        min_len = min(len(pred1), len(pred2), len(true_labels))

        correct1 = (pred1[:min_len] == true_labels[:min_len])
        correct2 = (pred2[:min_len] == true_labels[:min_len])

        b = np.sum(correct1 & ~correct2)
        c = np.sum(~correct1 & correct2)

        if b + c > 0:
            chi2_val = (abs(b - c) - 1)**2 / (b + c)
            p_val = 1 - chi2.cdf(chi2_val, 1)
        else:
            chi2_val = 0
            p_val = 1.0

        sig = "Yes*" if p_val < bonferroni_alpha else "No"
        print(f"{m1} vs {m2:<25} {chi2_val:>10.4f} {p_val:>12.6f} {sig:>8}")
        mcnemar_results[f"{m1}_vs_{m2}"] = {
            'chi2': round(chi2_val, 4), 'p_value': round(p_val, 6),
            'significant': p_val < bonferroni_alpha
        }

print(f"\nBonferroni-corrected \u03b1 = {bonferroni_alpha:.6f} (for {n_tests} comparisons)")
print(f"\nNote: Cross-tokenizer comparisons (XLM-R, mBERT, IndicBERT, ByT5) excluded")
print(f"because different tokenizers produce misaligned token sequences.")

In [ ]:
# =============================================================================
# CELL 5: FIX #3 — BOOTSTRAP CONFIDENCE INTERVALS (same-tokenizer only)
# =============================================================================
#
# ORIGINAL BUG: Computed bootstrap CIs for ALL models using BiLSTM's
# true_labels → cross-tokenizer models had misaligned predictions →
# XLM-R showed 70.13% CI instead of ~97.4%. Complete garbage.
#
# FIX: Only compute for MuRIL-tokenizer models where alignment is valid.
# =============================================================================

np.random.seed(42)
N_BOOTSTRAP = 1000

print("=" * 80)
print("CORRECTED BOOTSTRAP 95% CONFIDENCE INTERVALS")
print("=" * 80)
print(f"\n{'Model':<25} {'Mean F1':>10} {'95% CI Lower':>14} {'95% CI Upper':>14}")
print("-" * 65)

bootstrap_results = {}

for name in key_models:
    if name not in PREDICTIONS:
        continue

    if name not in MURIL_TOKENIZER_MODELS:
        # Cannot compute valid bootstrap — different tokenizer
        print(f"{name:<25} {'[skipped — different tokenizer]':>40}")
        continue

    preds = np.array(PREDICTIONS[name])
    min_len = min(len(preds), len(true_labels))
    preds = preds[:min_len]
    true = true_labels[:min_len]

    # Bootstrap
    boot_f1s = []
    n = len(true)
    for _ in range(N_BOOTSTRAP):
        idx = np.random.randint(0, n, size=n)
        boot_true = true[idx]
        boot_pred = preds[idx]
        f1 = f1_score(boot_true, boot_pred, average='macro', zero_division=0) * 100
        boot_f1s.append(f1)

    mean_f1 = np.mean(boot_f1s)
    ci_lower = np.percentile(boot_f1s, 2.5)
    ci_upper = np.percentile(boot_f1s, 97.5)

    print(f"{name:<25} {mean_f1:>9.2f}% {ci_lower:>13.2f}% {ci_upper:>13.2f}%")
    bootstrap_results[name] = {
        'mean_f1': round(mean_f1, 2),
        'ci_lower': round(ci_lower, 2),
        'ci_upper': round(ci_upper, 2)
    }

print(f"\n(Bootstrap: {N_BOOTSTRAP} samples, seed=42)")
print(f"Cross-tokenizer models skipped: their predictions don't align with stored true_labels.")

In [ ]:
# =============================================================================
# CELL 6: REPRINT ALL PAPER TABLES (for easy copy-paste)
# =============================================================================

# ---- TABLE 5: Baseline Results (unchanged — these were always correct) ----
print("=" * 90)
print("TABLE 5: BASELINE RESULTS (mean \u00b1 std over 3 seeds) — UNCHANGED")
print("=" * 90)
baselines = ['BiLSTM', 'SuffixCNN', 'MuRIL', 'XLM-R', 'mBERT', 'IndicBERT', 'ByT5']
print(f"{'Model':<15} {'Acc':>12} {'P':>12} {'R':>12} {'F1':>12} {'MCC':>10} {'Kappa':>10}")
print("-" * 85)
for name in baselines:
    ms = ALL_RESULTS[name]['mean_std']
    print(f"{name:<15} {ms['accuracy']:>12} {ms['precision']:>12} {ms['recall']:>12} {ms['f1']:>12} {ms['mcc']:>10} {ms['kappa']:>10}")

# ---- TABLE 6: Ablation (unchanged) ----
SUFFIX_LENGTHS = [3, 5, 7, 10]
FUSION_TYPES = ['concat', 'gated', 'attention', 'bilinear']

print(f"\n{'=' * 80}")
print("TABLE 6: HYBRID + CNN ABLATION (Macro-F1 %) — UNCHANGED")
print("=" * 80)
print(f"{'SL \\ Fusion':<15} {'concat':>14} {'gated':>14} {'attention':>14} {'bilinear':>14}")
print("-" * 75)
for sl in SUFFIX_LENGTHS:
    row = f"SL={sl:<10}"
    for ft in FUSION_TYPES:
        name = f"CNN-SL{sl}-{ft}"
        if name in ALL_RESULTS:
            row += f" {ALL_RESULTS[name]['mean_std']['f1']:>13}"
    print(row)

# ---- TABLE 7: Final Comparison (unchanged) ----
print(f"\n{'=' * 90}")
print("TABLE 7: FINAL MODEL COMPARISON — UNCHANGED")
print("=" * 90)
print(f"{'Model':<22} {'Type':<12} {'Acc':>14} {'F1':>14} {'MCC':>10} {'Params(M)':>10}")
print("-" * 85)
compare_models = baselines + [best_hybrid_name]
for name in compare_models:
    ms = ALL_RESULTS[name]['mean_std']
    mtype = 'Proposed' if name.startswith('CNN-SL') else 'Baseline'
    params_m = f"{ms['total_params']/1e6:.1f}M"
    marker = ' \u2605' if name == best_hybrid_name else ''
    print(f"{name:<22} {mtype:<12} {ms['accuracy']:>14} {ms['f1']:>14} {ms['mcc']:>10} {params_m:>10}{marker}")

# ---- TABLE 8: Cross-seed stability (unchanged) ----
print(f"\n{'=' * 80}")
print("TABLE 8: CROSS-SEED STABILITY — UNCHANGED")
print("=" * 80)
print(f"{'Model':<22} {'Seed 42':>10} {'Seed 123':>10} {'Seed 456':>10} {'\u03c3':>8}")
print("-" * 65)
for name in key_models:
    seed_keys = sorted([k for k in ALL_RESULTS[name] if k.startswith('seed_')])
    f1s = [ALL_RESULTS[name][sk]['test_metrics']['f1'] for sk in seed_keys]
    std = np.std(f1s)
    print(f"{name:<22} {f1s[0]:>9.2f}% {f1s[1]:>9.2f}% {f1s[2]:>9.2f}% {std:>7.3f}")

In [ ]:
# =============================================================================
# CELL 7: CORRECTED TABLE 9 — FOR PAPER (compact format)
# =============================================================================
# This is the key table that was WRONG before.
# Paper Table 9 only reports F1 per class, but now P and R are also correct.
# =============================================================================

print("=" * 70)
print("CORRECTED TABLE 9: PER-CLASS F1 (%) — FOR PAPER")
print("=" * 70)
print(f"\n{'Model':<22} {'SUB':>10} {'TIN':>10} {'OTH':>10}")
print("-" * 55)

for name in key_models:
    bs = ALL_RESULTS[name]['best_seed']
    pc = ALL_RESULTS[name][bs]['per_class']
    print(f"{name:<22} {pc['SUB']['f1']:>9.2f}  {pc['TIN']['f1']:>9.2f}  {pc['OTH']['f1']:>9.2f}")

print(f"\n--- Changes from original (if any): ---")
# The original had P=R=F1 which meant the F1 value = accuracy per class.
# If the confusion matrix is asymmetric (which it is), P and R will differ
# and F1 = 2*P*R/(P+R) will be slightly different from the old "accuracy" value.
print("Old values had P=R=F1=per-class accuracy (bug).")
print("New values have correct distinct P, R, and proper F1 = 2PR/(P+R).")
print("The F1 numbers may change slightly from what was in the paper.")
print("This is the CORRECT computation — update the paper with these values.")

In [ ]:
# =============================================================================
# CELL 8: TABLE 11 — ERROR ANALYSIS (unchanged, from confusion matrices)
# =============================================================================

print("=" * 80)
print("TABLE 11: ERROR ANALYSIS — UNCHANGED")
print("=" * 80)

error_models = ['IndicBERT', 'ByT5', best_hybrid_name]

print(f"\n{'Pattern':<15}", end="")
for m in error_models:
    print(f"{m:>18}", end="")
print()
print("-" * 70)

patterns = [
    ('SUB\u2192TIN', 0, 1), ('SUB\u2192OTH', 0, 2),
    ('TIN\u2192SUB', 1, 0), ('TIN\u2192OTH', 1, 2),
    ('OTH\u2192SUB', 2, 0), ('OTH\u2192TIN', 2, 1),
]

for pattern_name, true_cls, pred_cls in patterns:
    print(f"{pattern_name:<15}", end="")
    for m in error_models:
        bs = ALL_RESULTS[m]['best_seed']
        cm = ALL_RESULTS[m][bs]['confusion_matrix']
        print(f"{cm[true_cls][pred_cls]:>18}", end="")
    print()

In [ ]:
# =============================================================================
# CELL 9: TABLE 12 — COMPUTATIONAL COST (unchanged)
# =============================================================================

print("=" * 80)
print("TABLE 12: COMPUTATIONAL COST — UNCHANGED")
print("=" * 80)
print(f"\n{'Model':<22} {'Train(min)':>12} {'Tok/sec':>10} {'GPU(MB)':>10} {'Params(M)':>12}")
print("-" * 70)
for name in key_models:
    ms = ALL_RESULTS[name]['mean_std']
    params = f"{ms['total_params']/1e6:.1f}M"
    print(f"{name:<22} {ms['train_time_min']:>11.1f} {ms['inference_tps']:>10.0f} {ms['gpu_memory_mb']:>10.0f} {params:>12}")

In [ ]:
# =============================================================================
# CELL 10: SAVE CORRECTED JSON
# =============================================================================

OUTPUT_FILE = "experiment_results_FIXED.json"

# Update the complete results with fixes
complete_results['4_results'] = ALL_RESULTS

# Add fix metadata
complete_results['7_fixes_applied'] = {
    'fix_1_per_class': 'Recomputed P, R, F1 per class from confusion matrices (was P=R=F1 due to micro-average bug)',
    'fix_2_mcnemar': 'Restricted to same-tokenizer (MuRIL) model pairs only',
    'fix_3_bootstrap': 'Restricted to same-tokenizer models only (cross-tokenizer CIs were invalid)',
    'valid_mcnemar_pairs': list(mcnemar_results.keys()),
    'mcnemar_results': mcnemar_results,
    'bootstrap_results': bootstrap_results,
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(complete_results, f, indent=2, default=str)

print(f"\u2713 Corrected results saved to: {OUTPUT_FILE}")
print(f"  File size: {os.path.getsize(OUTPUT_FILE) / 1024:.0f} KB")

In [ ]:
# =============================================================================
# CELL 11: SUMMARY OF ALL CHANGES
# =============================================================================

print("\n" + "=" * 70)
print("  SUMMARY: WHAT CHANGED vs WHAT STAYED SAME")
print("=" * 70)

print("""
\u2705 UNCHANGED (paper values remain correct):
   - Table 4: Dataset statistics
   - Table 5: Baseline results (Acc, P, R, F1, MCC, Kappa)
   - Table 6: Ablation heatmap (all 16 hybrid F1 values)
   - Table 7: Final model comparison
   - Table 8: Cross-seed stability
   - Table 10: McNemar (paper correctly reported same-tokenizer only)
   - Table 11: Error analysis confusion counts
   - Table 12: Computational cost
   - All 7 figures

\u26a0\ufe0f  FIXED (update paper if needed):
   - Table 9: Per-class F1 values may change slightly
     Old: F1 = per-class accuracy (because P=R=F1 bug)
     New: F1 = 2*P*R/(P+R) computed correctly from confusion matrix
     Impact: Numbers change by ~0.01-0.5% depending on class asymmetry

\u2705 NOTEBOOK-ONLY FIXES (not in paper, no paper changes needed):
   - Bootstrap CIs: Now only computed for same-tokenizer models
   - McNemar in notebook: Now matches paper (same-tokenizer only)

\u27a1\ufe0f  ACTION REQUIRED:
   1. Compare old vs new Table 9 values above
   2. If F1 values changed, update Table 9 in paper
   3. Check if any claims about per-class performance need updating
   4. Replace experiment_results_complete.json with the FIXED version
""")

print("=" * 70)
print("  DONE! No retraining was needed.")
print("=" * 70)